# 🦠 Infection Spread — Coding Interview

**Format.** Live pair-programming style, ~45 min. Four parts, each builds on the last.
You implement one function per part. There are no partial-credit essays — the harness
grades you. Talk through your approach as you go.

**How to work this notebook**
1. Run **Cell 1** (the test harness) once. Don't edit it, don't read it — treat it as a black box.
2. Fill in the stubs in **Cell 2** and re-run it. The grader prints at the bottom.
3. Use `run_all(parts=(1,))` to focus on one part at a time while you're drilling.
4. **Gate:** you're "done" when Parts 1–3 are fully green. Part 4 is a stretch.

**Ground rules**
- 4-directional adjacency (up/down/left/right). No diagonals.
- Updates are **simultaneous** each day: cells infected *today* only start acting *tomorrow*.
- Grids can be empty, all-infected, or all-healthy. Handle the degenerate cases.
- The two `<400x400 grid>` cases exist to catch accidental O(days · cells²) blowups —
  aim for an efficient sweep, not a resimulation from scratch each day.

Read each part only when you get there — that's how it'd go in the room.

### Cell 1 — Test harness · run once, then ignore

In [ ]:
# CELL 1 — HARNESS
import time, copy, traceback, textwrap

TIME_LIMIT = 10.0

def _abbrev(x):
    if isinstance(x, list) and len(x) > 6:
        w = len(x[0]) if x and isinstance(x[0], list) else "?"
        return f"<{len(x)}x{w} grid>"
    return repr(x)

def _fmt_args(args):
    return "(" + ", ".join(_abbrev(a) for a in args) + ")"

def _grid(rows, cols, fill=0):
    return [[fill] * cols for _ in range(rows)]

def _p1_big():
    g = _grid(400, 400); g[0][0] = 1; return g

def _p2_big():
    g = _grid(400, 400)
    for r in range(400):
        g[r][200] = 2
    g[0][0] = 1
    return g

_P1 = [
    ("p1-01", ([[1]],), 0),
    ("p1-02", ([[0]],), -1),
    ("p1-03", ([],), 0),
    ("p1-04", ([[]],), 0),
    ("p1-05", ([[1, 0, 0, 0, 0]],), 4),
    ("p1-06", ([[0], [0], [1], [0]],), 2),
    ("p1-07", ([[0, 0, 0], [0, 1, 0], [0, 0, 0]],), 2),
    ("p1-08", ([[1, 0, 0], [0, 0, 0], [0, 0, 1]],), 2),
    ("p1-09", ([[0, 0, 0], [0, 0, 0], [0, 0, 0]],), -1),
    ("p1-10", ([[1, 1, 1], [1, 1, 1]],), 0),
    ("p1-11", ([[0]*5, [0]*5, [0, 0, 1, 0, 0], [0]*5, [0]*5],), 4),
    ("p1-12", ([[1, 0, 0, 1]],), 1),
    ("p1-13", ([[0, 1, 0], [0, 0, 0]],), 2),
    ("p1-14", (_p1_big(),), 798),
]

_P2 = [
    ("p2-01", ([[1, 2, 0]],), -1),
    ("p2-02", ([[0, 2, 0], [0, 2, 0], [0, 0, 1]],), 4),
    ("p2-03", ([[2, 2], [2, 2]],), 0),
    ("p2-04", ([[2]],), 0),
    ("p2-05", ([[0, 2]],), -1),
    ("p2-06", ([[1, 2], [2, 2]],), 0),
    ("p2-07", ([[1, 0, 2, 0]],), -1),
    ("p2-08", ([[2, 1, 2]],), 0),
    ("p2-09", ([[1, 0, 0], [0, 2, 0], [0, 0, 0]],), 4),
    ("p2-10", ([[0, 0, 0], [0, 0, 0], [0, 0, 0]],), -1),
    ("p2-11", ([[1, 2, 1]],), 0),
    ("p2-12", ([],), 0),
    ("p2-13", (_p2_big(),), -1),
]

_P3 = [
    ("p3-01", ([[1]], 1), 1),
    ("p3-02", ([[1, 0]], 1), 1),
    ("p3-03", ([[1, 0, 0]], 2), 4),
    ("p3-04", ([[0, 0]], 3), 0),
    ("p3-05", ([], 2), 0),
    ("p3-06", ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 3), 3),
    ("p3-07", ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 100), 102),
    ("p3-08", ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 1), 1),
    ("p3-09", ([[1, 0, 0, 0, 0]], 3), 7),
    ("p3-10", ([[1, 0], [0, 0]], 2), 4),
    ("p3-11", ([[1, 1], [1, 1]], 1), 1),
]

_P4 = [
    ("p4-01", ([[1, 0], [0, 0]], 1), 2),
    ("p4-02", ([[1, 0, 1], [0, 0, 0], [1, 0, 1]], 2), 2),
    ("p4-03", ([[1, 0, 0, 0]], 2), -1),
    ("p4-04", ([[1, 1], [1, 1]], 3), 0),
    ("p4-05", ([[0, 0], [0, 0]], 1), -1),
    ("p4-06", ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 4), 1),
    ("p4-07", ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 5), -1),
    ("p4-08", ([[1, 1, 0], [1, 0, 0], [0, 0, 0]], 2), -1),
    ("p4-09", ([[1]], 2), 0),
]

_PARTS = {
    1: ("Part 1 — basic spread",   "time_to_full_infection",              _P1),
    2: ("Part 2 — immune cells",   "time_to_full_infection_with_immunity", _P2),
    3: ("Part 3 — D-day recovery", "time_to_stable_state",                _P3),
    4: ("Part 4A — threshold K",   "time_to_full_infection_threshold",    _P4),
}

def _check(title, fname, cases):
    fn = globals().get(fname)
    print(f"\n=== {title}  [{fname}] ===")
    if fn is None:
        print("  function not defined — skipped")
        return None
    passed = 0
    for label, args, expected in cases:
        call = copy.deepcopy(args)
        t0 = time.perf_counter()
        try:
            got = fn(*call)
        except NotImplementedError:
            print(f"  {label}  SKIP (stub)")
            continue
        except Exception:
            print(f"  {label}  ERROR   args={_fmt_args(args)}")
            print(textwrap.indent(traceback.format_exc(limit=2), "      "))
            continue
        dt = time.perf_counter() - t0
        if got != expected:
            print(f"  {label}  FAIL    expected {expected}, got {got}")
            print(f"           args={_fmt_args(args)}")
        elif dt > TIME_LIMIT:
            print(f"  {label}  TOO SLOW  {dt:.2f}s > {TIME_LIMIT}s (correct value)")
        else:
            passed += 1
            print(f"  {label}  ok      {dt*1000:8.1f} ms")
    print(f"  --> {passed}/{len(cases)}")
    return passed, len(cases)

def run_all(parts=(1, 2, 3, 4)):
    res = {}
    for p in parts:
        title, fname, cases = _PARTS[p]
        res[p] = _check(title, fname, cases)
    print("\n" + "=" * 52)
    core = [p for p in (1, 2, 3) if p in res]
    clean = [p for p in core if res[p] and res[p][0] == res[p][1]]
    if len(clean) == 3:
        print("GATE: Parts 1-3 fully green. This pass is DONE.")
        if res.get(4) and res[4][0] == res[4][1]:
            print("      Part 4A also green — stretch cleared.")
    else:
        missing = [p for p in core if p not in clean]
        print(f"GATE: not there yet. Outstanding: Part(s) {missing}")
    print("=" * 52)

## The problem — read one part at a time

> **Part 1 — basic spread.**
> You've got an M×N grid representing a population. Each cell is healthy (`0`) or
> infected (`1`). Every day the infection spreads: any healthy cell adjacent to an
> infected cell becomes infected. Cells infected today don't start spreading until
> tomorrow — the update is simultaneous. There can be more than one source at the start.
> **Write a function that returns the number of days until the entire grid is infected.
> If that never happens, return `-1`.**
>
> → implement `time_to_full_infection(grid)`

---

> **Part 2 — immune cells.**
> Now some cells are immune (`2`): they never become infected and they never spread —
> they just sit there as walls. Same question: **how many days until every healthy cell
> is infected, or `-1` if some healthy cell is permanently walled off?**
>
> → implement `time_to_full_infection_with_immunity(grid)`

---

> **Part 3 — recovery.**
> Now add recovery. After a cell has been infected for `D` days it becomes immune — it
> stops spreading and can't be reinfected. **Recovery is applied *before* spread on the
> same day.** Return the number of days until the grid reaches a **stable state**: no
> actively infected cells left. Some cells may just stay healthy forever, and that's fine.
>
> → implement `time_to_stable_state(grid, D)`

---

> **Part 4 — threshold K.**
> Back to no recovery and no immunity, but change the spread rule: a healthy cell only
> becomes infected if it had **at least `K` infected neighbors** on the previous day.
> Same return: **days to full infection, or `-1`.**
>
> → implement `time_to_full_infection_threshold(grid, K)`

---

*Hints the interviewer will only give if you're stuck: Part 1 wants you to think about
what "day" means as a distance. Part 3's trap is the word "before." Part 4's `-1` isn't
about time running out.*

### Cell 2 — Your code · fill in the four stubs

In [ ]:
# ============================================================
# CELL 2 — YOUR CODE. Fill in the stubs. Results print below.
# ============================================================

def time_to_full_infection(grid: list[list[int]]) -> int:
    """
    grid: 0 = healthy, 1 = infected.
    Each day, every infected cell infects its 4-neighbors simultaneously.
    Cells infected today spread starting tomorrow.
    Returns days until all cells are infected, or -1 if impossible.
    """
    raise NotImplementedError


def time_to_full_infection_with_immunity(grid: list[list[int]]) -> int:
    """
    grid: 0 = healthy, 1 = infected, 2 = immune (permanent wall).
    Immune cells never infect and are never infected.
    Returns days until all healthy cells are infected,
    or -1 if any healthy cell is permanently unreachable.
    """
    raise NotImplementedError


def time_to_stable_state(grid: list[list[int]], D: int) -> int:
    """
    grid: 0 = healthy, 1 = infected on day 0.
    A cell infected on day t becomes immune once current_day - t >= D.
    Recovery is applied BEFORE spread on the same day.
    Returns the day on which no actively infected cells remain.
    """
    raise NotImplementedError


def time_to_full_infection_threshold(grid: list[list[int]], K: int) -> int:
    """
    grid: 0 = healthy, 1 = infected. No recovery, no immunity.
    A healthy cell becomes infected next day iff it had >= K
    infected 4-neighbors on the previous day.
    Returns days until all cells are infected, or -1 if impossible.
    """
    raise NotImplementedError


# ---- run ----
run_all(parts=(1, 2, 3, 4))   # narrow this, e.g. parts=(1,), while drilling